# Dijet JER systematic uncertainty

Compare the JER Up, Down, and default reconstructed dijet pseudorapidity distributions in embedding or Pythia MC. Full CM distributions are normalized to unit integral before comparison. Forward/Backward distributions are left unnormalized; their ratios always use standard independent-error propagation, never ROOT's binomial option.

Dedicated variation/default plots provide the signed shape variations used to estimate the JER systematic uncertainty.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dataclasses import replace
import math
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import COLORS, DEFAULT_PLOT_STYLE, save_canvas
from hist_analysis.python.systematic_fits import (
    calculate_bin_by_bin_systematic, fit_histogram_variations,
    format_fit_summary_lines, smooth_systematic_running_max,
    write_systematic_csv,
)


In [ ]:
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

`FULL_COMPARISON_RATIO_OPTION` controls errors on normalized Up/Def and Down/Def shape ratios. `FB_COMPARISON_RATIO_OPTION` controls errors only on the later ratio of two already-constructed F/B histograms. Set either to `''` for ROOT's standard propagation or `'B'` for option B. The construction of every F/B histogram is hard-coded to `''`. Full-distribution and F/B comparison ratios have separately configurable fit functions; both provisionally default to a first-order polynomial.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.4, 3.0)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS = tuple(TEST_DIJET_PTAVE_BINS)
REBIN_ETA = 2
FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: F / B is never binomial
FULL_COMPARISON_RATIO_OPTION = 'B'   # '' or 'B' for Up/Def and Down/Def
FB_COMPARISON_RATIO_OPTION = 'B'     # '' or 'B' for (F/B)_var / (F/B)_Def
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
SYSTEMATIC_Y_RANGE = (0., 1.0)       # percent units, e.g. (0.0, 10.0), or None
APPLY_SYSTEMATIC_SMOOTHING = True
SYSTEMATIC_COMBINATION = 'maximum'  # 'average' or 'maximum' Up/Down deviation
FULL_SMOOTHING_ORIGIN = -0.465 + 0.00001  # CM boost-shifted FindBin point
FULL_FIT_FUNCTION = 'pol4'    # provisional; likely final choice: pol2
FB_FIT_FUNCTION = 'pol2'      # provisional first-order polynomial
FULL_FIT_INITIAL_VALUES = {
    # 'Up / Def': (1.0, 0.0, 0.0),
    # 'Down / Def': (1.0, 0.0, 0.0),
    'Up / Def': (1.0, 0.0, 0.0, 0.0, 0.0),
    'Down / Def': (1.0, 0.0, 0.0, 0.0, 0.0),
}
FB_FIT_INITIAL_VALUES = {
    'Up / Def': (1.0, 0.0, 0.0),
    'Down / Def': (1.0, 0.0, 0.0),
}
FIT_OPTIONS = 'RQS0'          # fit range, result, quiet, no draw
FIT_WEIGHT_OPTION = ''       # 'W': weight 1 for each non-empty bin; '': use bin errors
EFFECTIVE_FIT_OPTIONS = FIT_OPTIONS + FIT_WEIGHT_OPTION
FIT_WEIGHT_TAG = 'weights1' if FIT_WEIGHT_OPTION == 'W' else 'weightsStd'
SYSTEMATIC_COMBINATION_TAG = (
    'systCombAve' if SYSTEMATIC_COMBINATION == 'average' else 'systCombMax'
)
OUTPUT_CONFIGURATION_TAG = (
    f'fullFit_{FULL_FIT_FUNCTION}_fbFit_{FB_FIT_FUNCTION}'
    f'_{FIT_WEIGHT_TAG}_{SYSTEMATIC_COMBINATION_TAG}'
)
SHOW_FIT_RESULTS = True
FIT_RESULTS_TEXT_SIZE = 0.018
FIT_RESULTS_BOX_BOUNDS = (0.43, 0.18, 0.88, 0.40)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_JER_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_JER',
))

CURVES = (
    DijetClosureCurve(
        'JER Up', 'hRecoDijetPtEtaCMJerUp_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerUp_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerUp_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JER Down', 'hRecoDijetPtEtaCMJerDown_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerDown_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerDown_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JER Default', 'hRecoDijetPtEtaCMJerDef_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerDef_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerDef_{eta_cut_index}',
    ),
)
NOMINAL = 'JER Default'
# Select entries from root_style.COLORS (0 red, 1 blue, 2 black, ...).
HISTOGRAM_COLOR_INDICES = {'JER Up': 0, 'JER Down': 1, 'JER Default': 2}
VARIATION_COLOR_INDICES = {'Up / Def': 0, 'Down / Def': 1}
SYSTEMATIC_COLOR_INDICES = {'JER bin-by-bin': 3, 'JER smoothed': 3}
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026, annotation_line_spacing=0.039,
    legend_text_size=0.028,
)
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError('GENERATOR must be embedding or pythia')
if DIRECTION not in ('pgoing', 'Pbgoing', 'combined'):
    raise ValueError('DIRECTION must be pgoing, Pbgoing, or combined')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('Forward/Backward construction must use standard errors')
if FIT_WEIGHT_OPTION not in ('', 'W'):
    raise ValueError("FIT_WEIGHT_OPTION must be empty or 'W'")
if SYSTEMATIC_COMBINATION not in ('average', 'maximum'):
    raise ValueError("SYSTEMATIC_COMBINATION must be 'average' or 'maximum'")
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
for label, color_index in (
    *HISTOGRAM_COLOR_INDICES.items(), *VARIATION_COLOR_INDICES.items(),
    *SYSTEMATIC_COLOR_INDICES.items(),
):
    if not isinstance(color_index, int) or not 0 <= color_index < len(COLORS):
        raise ValueError(f'Invalid root_style color index for {label}: {color_index}')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]

In [ ]:
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

DIRECTION_LABELS = {
    'pgoing': 'p-going', 'Pbgoing': 'Pb-going', 'combined': 'combined',
}
INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured MC ROOT file: {INPUT_FILE}')
INPUT_FILE

## Build projections and systematic comparisons

Each CM projection is scaled by `1 / Integral()` through `normalization='integral'`. Forward and backward projections are not normalized before division.

In [ ]:
jer_results = {}
eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
eta_cut_tag = int(round(10.0 * ETA_CUT))
systematic_output_tag = (
    'systematic_smoothed'
    if APPLY_SYSTEMATIC_SMOOTHING else 'systematic_nonsmoothed'
)

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    common_tag = (
        f'{GENERATOR}_{DIRECTION}_jerSystematics_etaCM_{eta_cut_tag}'
        f'_ptave_{ptave_tag}'
    )
    output_name = lambda plot: (
        f'{GENERATOR}_{DIRECTION}_jerSystematics_{plot}'
        f'_{OUTPUT_CONFIGURATION_TAG}'
        f'_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf'
    )
    eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
        INPUT_FILE, CURVES, eta_cut_index=ETA_CUT_INDEX,
        ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
        normalization='integral',
        ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
    )
    eta_variations = {
        ratio_label: ratio_to_nominal(
            eta_shapes[source_label], eta_shapes[NOMINAL],
            name=f'h_{common_tag}_{source_label.replace(" ", "_")}_to_default',
            option=FULL_COMPARISON_RATIO_OPTION,
        )
        for ratio_label, source_label in (
            ('Up / Def', 'JER Up'), ('Down / Def', 'JER Down'),
        )
    }
    fb_variations = {
        ratio_label: ratio_to_nominal(
            fb_ratios[source_label], fb_ratios[NOMINAL],
            name=f'h_{common_tag}_{source_label.replace(" ", "_")}_fb_to_default',
            option=FB_COMPARISON_RATIO_OPTION,
        )
        for ratio_label, source_label in (
            ('Up / Def', 'JER Up'), ('Down / Def', 'JER Down'),
        )
    }
    eta_fit_functions, eta_fit_summaries = fit_histogram_variations(
        eta_variations, formula=FULL_FIT_FUNCTION,
        fit_range=(-ETA_CUT, ETA_CUT),
        name_prefix=f'f_{common_tag}_full_ratio',
        fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FULL_FIT_INITIAL_VALUES,
    )
    fb_fit_functions, fb_fit_summaries = fit_histogram_variations(
        fb_variations, formula=FB_FIT_FUNCTION,
        fit_range=(0.0, ETA_CUT),
        name_prefix=f'f_{common_tag}_fb_ratio',
        fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FB_FIT_INITIAL_VALUES,
    )
    eta_systematic_unsmoothed = calculate_bin_by_bin_systematic(
        eta_variations['Up / Def'], eta_variations['Down / Def'],
        name=f'h_{common_tag}_full_jer_relative_systematic',
        up_function=eta_fit_functions['Up / Def'],
        down_function=eta_fit_functions['Down / Def'],
        evaluation_range=(-ETA_CUT, ETA_CUT),
        combination=SYSTEMATIC_COMBINATION,
    )
    fb_systematic_unsmoothed = calculate_bin_by_bin_systematic(
        fb_variations['Up / Def'], fb_variations['Down / Def'],
        name=f'h_{common_tag}_fb_jer_relative_systematic',
        up_function=fb_fit_functions['Up / Def'],
        down_function=fb_fit_functions['Down / Def'],
        evaluation_range=(0.0, ETA_CUT),
        combination=SYSTEMATIC_COMBINATION,
    )
    eta_systematic = (
        smooth_systematic_running_max(
            eta_systematic_unsmoothed,
            name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(-ETA_CUT, ETA_CUT),
            smoothing_origin=FULL_SMOOTHING_ORIGIN,
        )
        if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
    )
    fb_systematic = (
        smooth_systematic_running_max(
            fb_systematic_unsmoothed,
            name=f'{fb_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(0.0, ETA_CUT),
        )
        if APPLY_SYSTEMATIC_SMOOTHING else fb_systematic_unsmoothed
    )
    systematic_label = (
        'JER smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'JER bin-by-bin'
    )
    eta_systematic_percent = eta_systematic.Clone(
        f'{eta_systematic.GetName()}_percent'
    )
    eta_systematic_percent.SetDirectory(0)
    eta_systematic_percent.Scale(100.0)
    fb_systematic_percent = fb_systematic.Clone(
        f'{fb_systematic.GetName()}_percent'
    )
    fb_systematic_percent.SetDirectory(0)
    fb_systematic_percent.Scale(100.0)
    eta_systematics = {systematic_label: eta_systematic_percent}
    fb_systematics = {systematic_label: fb_systematic_percent}
    eta_fit_text = (
        format_fit_summary_lines(eta_fit_summaries)
        if SHOW_FIT_RESULTS else None
    )
    fb_fit_text = (
        format_fit_summary_lines(fb_fit_summaries)
        if SHOW_FIT_RESULTS else None
    )
    annotations = (
        GENERATOR.capitalize(),
        DIRECTION_LABELS[DIRECTION],
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} > 50 GeV',
        'p_{T}^{SubLead} > 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    canvases = {
        'eta_overlay': draw_overlay(
            eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
            y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.6,
            style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('full_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay',
        ),
        'eta_variations': draw_overlay(
            eta_variations, title='', x_title='#eta_{CM}^{dijet}',
            y_title='JER variation / default', x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            overlay_functions=eta_fit_functions,
            overlay_text=eta_fit_text,
            overlay_text_bounds=FIT_RESULTS_BOX_BOUNDS,
            overlay_text_size=FIT_RESULTS_TEXT_SIZE,
            style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('full_ratio_to_default'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_ratio',
        ),
        'fb_overlay': draw_overlay(
            fb_ratios, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Forward / Backward', x_range=fb_x_range,
            y_range=FB_RANGE, annotations=annotations, grid=DRAW_GRID,
            style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('fb_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay',
        ),
        'fb_variations': draw_overlay(
            fb_variations, title='', x_title='#eta_{CM}^{dijet}',
            y_title='(F/B)_{JER variation} / (F/B)_{default}',
            x_range=fb_x_range, y_range=FB_DOUBLE_RATIO_RANGE,
            reference_y=1.0, annotations=annotations, grid=DRAW_GRID,
            overlay_functions=fb_fit_functions,
            overlay_text=fb_fit_text,
            overlay_text_bounds=FIT_RESULTS_BOX_BOUNDS,
            overlay_text_size=FIT_RESULTS_TEXT_SIZE,
            style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name('fb_ratio_to_default'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_ratio',
        ),
        'eta_systematic': draw_overlay(
            eta_systematics, title='', x_title='#eta_{CM}^{dijet}',
            y_title='JER Rel. Syst. Uncrt. (%)',
            x_range=eta_x_range, y_range=SYSTEMATIC_Y_RANGE,
            annotations=annotations, grid=DRAW_GRID,
            show_legend=False,
            style_indices=SYSTEMATIC_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name(
                f'{systematic_output_tag}_full_relative'
            ),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_systematic',
        ),
        'fb_systematic': draw_overlay(
            fb_systematics, title='', x_title='#eta_{CM}^{dijet}',
            y_title='JER Rel. Syst. Uncrt. (%)',
            x_range=fb_x_range, y_range=SYSTEMATIC_Y_RANGE,
            annotations=annotations, grid=DRAW_GRID,
            show_legend=False,
            style_indices=SYSTEMATIC_COLOR_INDICES, style=PLOT_STYLE,
            output=OUTPUT_DIR / output_name(
                f'{systematic_output_tag}_fb_relative'
            ),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_systematic',
        ),
    }
    jer_results[ptave_range] = {
        'eta_shapes': eta_shapes, 'eta_variations': eta_variations,
        'forward_backward': fb_ratios, 'fb_variations': fb_variations,
        'eta_systematic': eta_systematic,
        'fb_systematic': fb_systematic,
        'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
        'fb_systematic_unsmoothed': fb_systematic_unsmoothed,
        'systematic_smoothing_applied': APPLY_SYSTEMATIC_SMOOTHING,
        'systematic_combination': SYSTEMATIC_COMBINATION,
        'full_smoothing_origin': FULL_SMOOTHING_ORIGIN,
        'eta_systematic_percent': eta_systematic_percent,
        'fb_systematic_percent': fb_systematic_percent,
        'eta_fit_functions': eta_fit_functions,
        'eta_fit_summaries': eta_fit_summaries,
        'fb_fit_functions': fb_fit_functions,
        'fb_fit_summaries': fb_fit_summaries,
        'keys': selected_keys, 'canvases': canvases,
    }
    print(ptave_range, selected_keys)
    for canvas in canvases.values():
        display(canvas)

## Validation summary

Check the unit-integral normalization, report the finite extrema of the signed systematic ratios, and print the fitted coefficients with their uncertainties and fit quality. The envelope per bin can be read as `max(abs(Up/Def - 1), abs(Down/Def - 1))`.

In [ ]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None

for ptave_range, result in jer_results.items():
    print(f'\npTave interval {ptave_range}, eta cut {ETA_CUT:g}')
    print('CM integral normalization:', {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    })
    print('CM variation/default ranges:', {
        label: finite_nonzero_range(histogram)
        for label, histogram in result['eta_variations'].items()
    })
    print('F/B variation/default ranges:', {
        label: finite_nonzero_range(histogram)
        for label, histogram in result['fb_variations'].items()
    })
    print('Bin-by-bin relative systematic ranges:', {
        'CM': finite_nonzero_range(result['eta_systematic']),
        'F/B': finite_nonzero_range(result['fb_systematic']),
    })
    for observable, summaries in (
        ('CM variation/default', result['eta_fit_summaries']),
        ('F/B variation/default', result['fb_fit_summaries']),
    ):
        print(f'{observable} fits:')
        for label, summary in summaries.items():
            coefficients = ', '.join(
                f'p{index}={value:.6g} +/- {error:.3g}'
                for index, (value, error) in enumerate(zip(
                    summary['parameters'], summary['parameter_errors'],
                ))
            )
            print(
                f"  {label}: {summary['formula']}, {coefficients}, "
                f"chi2/ndf={summary['chi2']:.3g}/{summary['ndf']}, "
                f"prob={summary['probability']:.3g}"
            )

In [ ]:
# Overlay ratio-to-default curves with the symmetric evaluated systematic band.
SHOW_FIT_LINES_ON_SYSTEMATIC_BANDS = True  # set False to hide fitted curves
# The retained systematic histograms are fractional: 0.0002 corresponds to 0.02%.
def draw_ratio_with_systematic_band(ratios, systematic, *, x_range, y_range, fit_functions,
                                    output, canvas_name, x_title, annotations):
    canvas = draw_overlay(
        ratios, title='', x_title=x_title, y_title='Variation / Default',
        x_range=x_range, y_range=y_range, reference_y=1.0,
        annotations=annotations, grid=DRAW_GRID, overlay_functions=fit_functions,
        style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
        output=None, save_png=False, canvas_name=canvas_name,
    )
    graph = ROOT.TGraphAsymmErrors(systematic.GetNbinsX())
    for bin_index in range(1, systematic.GetNbinsX() + 1):
        point = bin_index - 1
        graph.SetPoint(point, systematic.GetBinCenter(bin_index), 1.0)
        graph.SetPointError(
            point, systematic.GetBinWidth(bin_index) / 2.0,
            systematic.GetBinWidth(bin_index) / 2.0,
            systematic.GetBinContent(bin_index), systematic.GetBinContent(bin_index),
        )
    graph.SetFillColorAlpha(COLORS[3], 0.30)
    graph.SetLineColor(COLORS[3])
    # graph.Draw('3 SAME')
    graph.Draw('E2')
    for ratio in ratios.values():
        ratio.Draw('E1 SAME')
    if SHOW_FIT_LINES_ON_SYSTEMATIC_BANDS:
        for function in fit_functions.values():
            function.Draw('SAME')
    canvas._overlay_objects[0].AddEntry(graph, 'Syst. Uncrt.', 'f')
    canvas.Modified(); canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    canvas._overlay_objects.append(graph)
    return canvas

for ptave_range, result in jer_results.items():
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    common_tag = f'{GENERATOR}_{DIRECTION}_jerSystematics_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
    output_prefix = f'{GENERATOR}_{DIRECTION}_jerSystematics'
    annotations = (GENERATOR.capitalize(), DIRECTION_LABELS[DIRECTION], f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV')
    result['eta_ratio_systematic_band'] = draw_ratio_with_systematic_band(
        result['eta_variations'], result['eta_systematic'], fit_functions=result['eta_fit_functions'], x_range=eta_x_range,
        y_range=FULL_RATIO_RANGE, output=OUTPUT_DIR / f'{output_prefix}_full_ratio_with_systematic_band_{OUTPUT_CONFIGURATION_TAG}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf',
        canvas_name=f'{common_tag}_full_ratio_with_systematic_band', x_title='#eta_{CM}^{dijet}', annotations=annotations,
    )
    result['fb_ratio_systematic_band'] = draw_ratio_with_systematic_band(
        result['fb_variations'], result['fb_systematic'], fit_functions=result['fb_fit_functions'], x_range=fb_x_range,
        y_range=FB_DOUBLE_RATIO_RANGE, output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_with_systematic_band_{OUTPUT_CONFIGURATION_TAG}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf',
        canvas_name=f'{common_tag}_fb_ratio_with_systematic_band', x_title='#eta_{CM}^{dijet}', annotations=annotations,
    )

    # Headerless CSV in TGraphErrors order: eta_center, 1.0, eta_half_width, fractional uncertainty.
    for observable, systematic, acceptance in (('full', result['eta_systematic'], (-ETA_CUT, ETA_CUT)), ('fb', result['fb_systematic'], (0.0, ETA_CUT))):
        csv_path = write_systematic_csv(
            systematic,
            OUTPUT_DIR / f'{output_prefix}_{systematic_output_tag}_{observable}_relative_{OUTPUT_CONFIGURATION_TAG}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.csv',
            evaluation_range=acceptance,
        )
        result[f'{observable}_systematic_csv'] = csv_path